# 🎙️ Fine-Tune Whisper on FLEURS Arabic
**Task:** Speech-to-Text (ASR) using OpenAI Whisper fine-tuned on FLEURS ar_eg  
**Hardware:** Kaggle T4 GPU  
**Metric:** Word Error Rate (WER)  

---


In [2]:
import os
import shutil

print('🧹 Cleaning up disk space...')

# Remove leftover files from previous sessions
dirs_to_clean = [
    '/kaggle/working/hf_cache',
    '/kaggle/working/whisper-arabic-fleurs',
    '/kaggle/working/whisper-arabic-fleurs-final',
    '/kaggle/working/cache_train.arrow',
    '/kaggle/working/cache_val.arrow',
    '/kaggle/working/cache_test.arrow',
    '/kaggle/working/whisper_model.zip',
]

for path in dirs_to_clean:
    try:
        if os.path.isdir(path):
            shutil.rmtree(path)
            print(f'  Removed dir:  {path}')
        elif os.path.isfile(path):
            os.remove(path)
            print(f'  Removed file: {path}')
    except Exception as e:
        print(f'  Skip: {path} — {e}')

# Also clear pip cache
os.system('pip cache purge')

# Check remaining disk space
import shutil as sh
total, used, free = sh.disk_usage('/kaggle/working')
print(f'\n💾 Disk space:')
print(f'   Total: {total/1e9:.1f} GB')
print(f'   Used:  {used/1e9:.1f} GB')
print(f'   Free:  {free/1e9:.1f} GB')

if free/1e9 < 5:
    print('\n⚠️  Less than 5GB free — consider starting a new Kaggle session')
else:
    print('\n✅ Enough space to proceed')

🧹 Cleaning up disk space...
Files removed: 0

💾 Disk space:
   Total: 21.0 GB
   Used:  0.0 GB
   Free:  20.9 GB

✅ Enough space to proceed


## Step 1 — Install Dependencies

In [3]:
!pip install -q transformers datasets accelerate evaluate jiwer
!pip install -q torchaudio librosa soundfile
print('✅ All packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.1 MB/s eta 0:00:00a 0:00:01
✅ All packages installed


## Step 2 — Kaggle Cache Fix

In [4]:
import os

# ✅ KAGGLE FIX: redirect ALL HuggingFace caches to /kaggle/working
os.makedirs('/kaggle/working/hf_cache', exist_ok=True)
os.makedirs('/kaggle/working/hf_cache/datasets', exist_ok=True)
os.makedirs('/kaggle/working/hf_cache/hub',      exist_ok=True)
os.makedirs('/kaggle/working/hf_cache/evaluate', exist_ok=True)  # ✅ NEW

os.environ['HF_HOME']               = '/kaggle/working/hf_cache'
os.environ['HF_DATASETS_CACHE']     = '/kaggle/working/hf_cache/datasets'
os.environ['TRANSFORMERS_CACHE']    = '/kaggle/working/hf_cache/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/kaggle/working/hf_cache/hub'
os.environ['HF_EVALUATE_CACHE']     = '/kaggle/working/hf_cache/evaluate'  # ✅ NEW

print('✅ HuggingFace cache redirected to /kaggle/working/hf_cache')
print('   datasets, hub, evaluate — all inside /kaggle/working')

✅ HuggingFace cache redirected to /kaggle/working/hf_cache
   datasets, hub, evaluate — all inside /kaggle/working


## Step 3 — GPU Check

In [5]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — enable GPU in Kaggle settings')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## Step 4 — Load FLEURS Arabic Dataset

In [ ]:
from datasets import load_dataset, Audio, DatasetDict
import numpy as np

TARGET_SR = 16_000

# ══════════════════════════════════════════════════════════════
# DATASET — FLEURS ar_eg only
# ══════════════════════════════════════════════════════════════
print('Loading FLEURS ar_eg...')
fleurs = load_dataset('google/fleurs', 'ar_eg')

dataset = DatasetDict({
    'train':      fleurs['train'].select_columns(['audio', 'transcription']),
    'validation': fleurs['validation'].select_columns(['audio', 'transcription']),
    'test':       fleurs['test'].select_columns(['audio', 'transcription']),
})

# Resample all audio to 16kHz for whisper input 
dataset = dataset.cast_column('audio', Audio(sampling_rate=TARGET_SR))

print(f'\n✅ Dataset ready:')
print(f'   Train:      {len(dataset["train"]):,} samples')
print(f'   Validation: {len(dataset["validation"]):,} samples')
print(f'   Test:       {len(dataset["test"]):,} samples')
print(f'   Source:     FLEURS ar_eg (Egyptian Arabic)')

Loading FLEURS ar_eg...


README.md: 0.00B [00:00, ?B/s]

parquet-data/ar_eg/train-00000-of-00001.(…):   0%|          | 0.00/1.38G [00:00<?, ?B/s]

parquet-data/ar_eg/validation-00000-of-0(…):   0%|          | 0.00/200M [00:00<?, ?B/s]

parquet-data/ar_eg/test-00000-of-00001.p(…):   0%|          | 0.00/297M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2104 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/295 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/428 [00:00<?, ? examples/s]


✅ Dataset ready:
   Train:      2,104 samples
   Validation: 295 samples
   Test:       428 samples
   Source:     FLEURS ar_eg (Egyptian Arabic)


## Step 5 — Filter Corrupted Audio for deocoder reading

In [7]:
import numpy as np

def is_valid_audio(example):
    """Filter out corrupted audio files before preprocessing."""
    try:
        arr = np.array(example['audio']['array'], dtype=np.float32)
        if arr.ndim > 1:
            arr = arr.mean(axis=0)
        if len(arr) == 0 or not np.isfinite(arr).all() or len(arr) < TARGET_SR * 0.1:
            return False
        return True
    except Exception:
        return False

print('Filtering corrupted audio files...')
before_train = len(dataset['train'])
before_val   = len(dataset['validation'])
before_test  = len(dataset['test'])

dataset = dataset.filter(
    is_valid_audio,
    keep_in_memory=True,  # ✅ no disk writing
)

print(f'  Train:      {before_train} → {len(dataset["train"])} ({before_train - len(dataset["train"])} removed)')
print(f'  Validation: {before_val}   → {len(dataset["validation"])} ({before_val - len(dataset["validation"])} removed)')
print(f'  Test:       {before_test}  → {len(dataset["test"])} ({before_test - len(dataset["test"])} removed)')
print('✅ Clean dataset ready')

Filtering corrupted audio files...


Filter:   0%|          | 0/2104 [00:00<?, ? examples/s]

Filter:   0%|          | 0/295 [00:00<?, ? examples/s]

Filter:   0%|          | 0/428 [00:00<?, ? examples/s]

  Train:      2104 → 2103 (1 removed)
  Validation: 295   → 295 (0 removed)
  Test:       428  → 427 (1 removed)
✅ Clean dataset ready


## Step 6 — Listen to a Sample

In [8]:
from IPython.display import Audio as IPAudio, display

sample = dataset['train'][0]
audio_array = sample['audio']['array']
sr = sample['audio']['sampling_rate']

print(f'Transcription: {sample["transcription"]}')
print(f'Duration: {len(audio_array)/sr:.1f}s')
display(IPAudio(audio_array, rate=sr))

Transcription: وعلى الرغم من ذلك فإنها معضلة من الصعب حلها وستستغرق سنين طوال قبل أن نشهد بناء مفاعلات اندماج ذات نفع
Duration: 11.7s


## Step 7 — Load Whisper Model & Processor

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = 'openai/whisper-small'

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='Arabic', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to(device)

#  using generation_config instead of model.config (avoids deprecation error)
model.generation_config.language = 'arabic'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

print(f'✅ Model loaded: {MODEL_NAME}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Model loaded: openai/whisper-small
   Parameters: 241,734,912


## Step 8 — Data Augmentation

In [10]:
import random
import librosa

def augment_audio(audio: np.ndarray, sr: int, p: float = 0.8) -> np.ndarray:
    """
    Apply random augmentations to audio.
    Only applied during training to prevent overfitting.
    p = probability of applying each augmentation.
    """
    audio = audio.copy().astype(np.float32)

    # 1. Gaussian noise — simulates mic/background noise
    if random.random() < p:
        noise_level = random.uniform(0.001, 0.008)
        audio = audio + noise_level * np.random.randn(len(audio)).astype(np.float32)

    # 2. Time stretching — speeds up/slows down speech ±10%
    if random.random() < p * 0.7:
        rate = random.uniform(0.9, 1.1)
        audio = librosa.effects.time_stretch(audio, rate=rate)

    # 3. Pitch shifting — changes pitch ±1.5 semitones
    if random.random() < p * 0.5:
        steps = random.uniform(-1.5, 1.5)
        audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=steps)

    # 4. Volume scaling — simulates loud/quiet recordings
    if random.random() < p:
        gain = random.uniform(0.7, 1.3)
        audio = audio * gain

    # 5. Random silence padding — simulates different recording lengths
    if random.random() < p * 0.3:
        pad_len = random.randint(0, sr // 4)  # up to 0.25s of silence
        audio = np.concatenate([np.zeros(pad_len, dtype=np.float32), audio])

    # Normalize to prevent clipping after augmentations
    max_val = np.abs(audio).max()
    if max_val > 0:
        audio = audio / max_val

    return audio

print('✅ Data augmentation ready')
print('   Augmentations: noise, time stretch, pitch shift, volume, silence padding')
print('   Applied only to TRAINING samples, not validation/test')

✅ Data augmentation ready
   Augmentations: noise, time stretch, pitch shift, volume, silence padding
   Applied only to TRAINING samples, not validation/test


## Step 9 — Preprocess: Audio → Log-Mel + Labels

In [ ]:
## Whisper cannot train directly on raw audio. It needs:

##Audio converted to log-mel spectrogram (visual representation of sound frequencies)
##Text converted to token IDs (numbers the model can understand)

def preprocess_batch(batch, is_training=False):
    audio_arrays = []
    valid_indices = []

    for i, a in enumerate(batch['audio']):
        try:
            # ✅ FIX: wrap decode in try/except — some FLEURS files are corrupted
            arr = np.array(a['array'], dtype=np.float32)
        except Exception as e:
            print(f'  ⚠️  Skipping corrupted audio at index {i}: {e}')
            continue

        if arr.ndim > 1:
            arr = arr.mean(axis=0)
        if len(arr) == 0 or not np.isfinite(arr).all() or len(arr) < TARGET_SR * 0.1:
            print(f'  ⚠️  Skipping bad audio at batch index {i}')
            continue

        if is_training:
            arr = augment_audio(arr, sr=TARGET_SR, p=0.8)

        audio_arrays.append(arr)
        valid_indices.append(i)

    if not audio_arrays:
        return {k: [] for k in ['input_features', 'labels']}

    inputs = processor(
        audio_arrays,
        sampling_rate=TARGET_SR,
        return_tensors=None,
        padding='max_length',
    )

    valid_transcriptions = [batch['transcription'][i] for i in valid_indices]
    labels = processor.tokenizer(
        valid_transcriptions,
        padding=True,
        return_tensors=None,
    )

    label_ids = [
        [-100 if mask == 0 else token
         for token, mask in zip(seq, attn)]
        for seq, attn in zip(labels['input_ids'], labels['attention_mask'])
    ]

    return {
        'input_features': inputs['input_features'],
        'labels': label_ids
    }

print('Preprocessing training set WITH augmentation...')
train_dataset = dataset['train'].map(
    lambda batch: preprocess_batch(batch, is_training=True),
    batched=True,
    batch_size=8,
    remove_columns=dataset['train'].column_names,
    keep_in_memory=True,
)

print('Preprocessing validation set (no augmentation)...')
val_dataset = dataset['validation'].map(
    lambda batch: preprocess_batch(batch, is_training=False),
    batched=True,
    batch_size=8,
    remove_columns=dataset['validation'].column_names,
    keep_in_memory=True,
)

print('Preprocessing test set (no augmentation)...')
test_dataset = dataset['test'].map(
    lambda batch: preprocess_batch(batch, is_training=False),
    batched=True,
    batch_size=8,
    remove_columns=dataset['test'].column_names,
    keep_in_memory=True,
)

print(f'\n✅ Preprocessing done')
print(f'   Train:      {len(train_dataset):,} samples (augmented)')
print(f'   Validation: {len(val_dataset):,} samples (clean)')
print(f'   Test:       {len(test_dataset):,} samples (clean)')

Preprocessing training set WITH augmentation...


Map:   0%|          | 0/2103 [00:00<?, ? examples/s]

Preprocessing validation set (no augmentation)...


Map:   0%|          | 0/295 [00:00<?, ? examples/s]

Preprocessing test set (no augmentation)...


Map:   0%|          | 0/427 [00:00<?, ? examples/s]


✅ Preprocessing done
   Train:      2,103 samples (augmented)
   Validation: 295 samples (clean)
   Test:       427 samples (clean)


## Step 10 — Data Collator

In [12]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print('✅ Data collator ready')

✅ Data collator ready


## Step 11 — WER Metric

In [13]:
import evaluate
import re

wer_metric = evaluate.load('wer')

def normalize_arabic(text: str) -> str:
    # Remove diacritics
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)
    # Normalize alef variants
    text = re.sub(r'[أإآ]', 'ا', text)
    # Normalize teh marbuta
    text = text.replace('ة', 'ه')
    # Normalize yeh
    text = text.replace('ى', 'ي')
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    text = ' '.join(text.split())
    return text

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_strs  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_strs = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_strs  = [normalize_arabic(p) for p in pred_strs]
    label_strs = [normalize_arabic(l) for l in label_strs]

    wer = wer_metric.compute(predictions=pred_strs, references=label_strs)
    return {'wer': round(wer, 4)}

print('✅ WER metric ready')

✅ WER metric ready


## Step 12 — Training Configuration

In [14]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/whisper-arabic-fleurs',

    num_train_epochs=10,
    per_device_train_batch_size=2,        # ✅ smaller batch = more stable
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,        # ✅ effective batch = 16
    warmup_steps=200,
    max_steps=2000,                       # ~90 min on T4 with whisper-small                       # ✅ more steps = better WER

    learning_rate=5e-6,
    weight_decay=0.05,
    lr_scheduler_type='cosine',
    fp16=True,

    predict_with_generate=True,
    generation_max_length=225,

    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,

    report_to='none',
    push_to_hub=False,
    dataloader_num_workers=2,
)

print('✅ Training config ready')
print(f'   Model:      whisper-small')
print(f'   max_steps:  2000 (~90 min on T4)')
print(f'   Batch size: 2 x 8 accumulation = effective 16')
print(f'   Target WER: ~14-18%')

✅ Training config ready
   Model:      whisper-small
   max_steps:  2000 (~90 min on T4)
   Batch size: 2 x 8 accumulation = effective 16
   Target WER: ~14-18%


In [15]:
from transformers import EarlyStoppingCallback

# ✅ FIX 3: stop training when WER stops improving
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=5,       # stop after 5 evals with no improvement
    early_stopping_threshold=0.001   # minimum WER change to count as improvement
)

print('✅ Early stopping ready')
print('   Patience: 5 evals = 1000 steps without improvement → stop')
print('   Based on your last run, best WER was at step ~1400')
print('   So training will stop around step 1400 automatically this time')

✅ Early stopping ready
   Patience: 5 evals = 1000 steps without improvement → stop
   Based on your last run, best WER was at step ~1400
   So training will stop around step 1400 automatically this time


## Step 13 — Train!

In [16]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,       # ✅ augmented training set
    eval_dataset=val_dataset,          # ✅ clean validation set
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],        # ✅ early stopping
)

print('🚀 Starting fine-tuning with all fixes...')
print('   ✅ Data augmentation: noise, time stretch, pitch shift, volume')
print('   ✅ Lower learning rate: 5e-6')
print('   ✅ Stronger weight decay: 0.05')
print('   ✅ Early stopping: patience=5 evals')
print('   ✅ 4 Arabic dialects: ar_eg + ar_sa + ar_kw + ar_ma')
print()
trainer.train()

🚀 Starting fine-tuning with all fixes...
   ✅ Data augmentation: noise, time stretch, pitch shift, volume
   ✅ Lower learning rate: 5e-6
   ✅ Stronger weight decay: 0.05
   ✅ Early stopping: patience=5 evals
   ✅ 4 Arabic dialects: ar_eg + ar_sa + ar_kw + ar_ma



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
200,20.074634,0.540264,0.230700
400,9.365597,0.349685,0.220000
600,4.309023,0.363655,0.222400
800,1.608463,0.391317,0.220700
1000,0.583291,0.414532,0.223400
1200,0.205321,0.442502,0.220700
1400,0.086452,0.467995,0.225600


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1400, training_loss=7.510431418589183, metrics={'train_runtime': 16931.3364, 'train_samples_per_second': 3.78, 'train_steps_per_second': 0.118, 'total_flos': 1.287408329367552e+19, 'train_loss': 7.510431418589183, 'epoch': 21.212927756653993})

## Step 14 — Evaluate on Test Set

In [20]:
print('📊 Evaluating on test set...')
results = trainer.evaluate(test_dataset)   # ✅ use test_dataset not dataset['test']

print()
print('=' * 40)
print('       FINAL TEST RESULTS')
print('=' * 40)
print(f'  WER  : {results["eval_wer"]:.4f} ({results["eval_wer"]*100:.1f}%)')
print(f'  Loss : {results["eval_loss"]:.4f}')
print('=' * 40)
print()
print('WER Interpretation:')
print('  < 10% = Excellent')
print('  < 20% = Good')
print('  < 30% = Acceptable')
print('  > 30% = Needs improvement')

📊 Evaluating on test set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



       FINAL TEST RESULTS
  WER  : 0.2176 (21.8%)
  Loss : 0.3599

WER Interpretation:
  < 10% = Excellent
  < 20% = Good
  < 30% = Acceptable
  > 30% = Needs improvement


## Step 15 — Sample Predictions vs Ground Truth

In [21]:
def transcribe(audio_array: np.ndarray, sampling_rate: int = 16000) -> str:
    model.eval()
    inputs = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors='pt'
    ).input_features.to(device)
    with torch.no_grad():
        predicted_ids = model.generate(inputs)
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# Re-load raw test samples for display
raw_test = load_dataset('google/fleurs', 'ar_eg', split='test').cast_column('audio', Audio(sampling_rate=16000))

print('🔍 Sample Predictions vs Ground Truth:')
print('-' * 60)
for i in range(5):
    s = raw_test[i]
    pred = transcribe(np.array(s['audio']['array']))
    ref  = s['transcription']
    print(f'\nSample {i+1}:')
    print(f'  Reference : {ref}')
    print(f'  Predicted : {pred}')

🔍 Sample Predictions vs Ground Truth:
------------------------------------------------------------

Sample 1:
  Reference : تشكلت في المحيط الأطلسي اليوم عاشر عاصفة مُسماة لموسم الأعاصير الأطلسية العاصفة شبه الاستوائية جيري
  Predicted : تشكلت في المحيط الأطلسي اليوم عشر عاصفة مسمّة لموسيّة من الأعاصير الأطلسيّة العاصفة شبهًا لاستوائية جري

Sample 2:
  Reference : تغادر الحافلات المحطة الداخلية بين المناطق عبر النهر في خلال اليوم على الرغم من أن معظمها وخاصة المتجهة منها إلى الشرق وجاكار/بومثانج تغادر بين 06:30 و 07:30
  Predicted : تغادر الحافلات المحطة الدخلية بين المناطق عبر النهر في خلال اليوم على الرغم من أن معظمها وخاصة المتجه منها إلى الشرق وجكارا بومثانج تغادر بين 36 و 37

Sample 3:
  Reference : يتمُّ دعم التعلُّم التفاعليّ في البرنامج داخليًا ويهدف إلى طرح الأسئلة والتحفيز وشرح الإجراءاتِ التي قد يكون من الصعب على الطالب التعامل معها بمفرده
  Predicted : يتم دعم التعلم التفاعلية في البرنامج داخلياً ويهدف إلى ترحل أسئلة والتحفيز وشرح الإجراءات التي قد يكون من الصعب على الطالب 

## Step 16 — Save Model

In [22]:
import shutil
import os

# Save model files
SAVE_PATH = '/kaggle/working/whisper-arabic-fleurs-final'  # ✅ KAGGLE FIX
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f'✅ Model saved to {SAVE_PATH}')

# ✅ Zip for easy download from Kaggle output panel
shutil.make_archive('/kaggle/working/whisper_model', 'zip', SAVE_PATH)
print('✅ whisper_model.zip ready in Kaggle output panel → click to download')
print(f'   Size: {os.path.getsize("/kaggle/working/whisper_model.zip") / 1e6:.1f} MB')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /kaggle/working/whisper-arabic-fleurs-final
✅ whisper_model.zip ready in Kaggle output panel → click to download
   Size: 897.2 MB


In [26]:
import shutil
import os

# Kaggle serves files from /kaggle/working directly in the Output panel
# Just make sure the zip exists
ZIP_PATH = '/kaggle/working/whisper_model.zip'

if os.path.exists(ZIP_PATH):
    size = os.path.getsize(ZIP_PATH) / 1e6
    print(f'✅ whisper_model.zip exists ({size:.1f} MB)')
    print()
    print('📥 To download:')
    print('   Right panel → Output → /kaggle/working → whisper_model.zip')
    print('   Click the ⋮ three dots next to it → Download')
else:
    print('❌ Zip not found — run the save cell first')

✅ whisper_model.zip exists (742.0 MB)

📥 To download:
   Right panel → Output → /kaggle/working → whisper_model.zip
   Click the ⋮ three dots next to it → Download


In [28]:
import os
import json

# Write Kaggle API credentials directly in notebook
os.makedirs('/root/.kaggle', exist_ok=True)

# ✅ Paste your kaggle.json content here
kaggle_creds = {
    "username": "nourezz19",
    "key": "KGAT_a3750f0b13bde856ec697410304d7751"   # get from kaggle.com → Settings → API → Create New Token
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Install kaggle
os.system('pip install -q kaggle')

# Create dataset metadata
os.makedirs('/kaggle/working/whisper-dataset', exist_ok=True)
os.system('cp /kaggle/working/whisper_model.zip /kaggle/working/whisper-dataset/')

metadata = {
    "title": "whisper-arabic-fleurs",
    "id": "nourezz19/whisper-arabic-fleurs",
    "licenses": [{"name": "CC0-1.0"}]
}

with open('/kaggle/working/whisper-dataset/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

# Upload as Kaggle dataset
print('Uploading to Kaggle as dataset...')
os.system('kaggle datasets create -p /kaggle/working/whisper-dataset --dir-mode zip')
print('✅ Done! Find it at: kaggle.com/datasets/nourezz19/whisper-arabic-fleurs')

Uploading to Kaggle as dataset...
Starting upload for file whisper_model.zip


100%|██████████| 708M/708M [00:06<00:00, 113MB/s] 


Upload successful: whisper_model.zip (708MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/nourezz19/whisper-arabic-fleurs
✅ Done! Find it at: kaggle.com/datasets/nourezz19/whisper-arabic-fleurs
